# Preparación de datos

**Conjunto de datos:** Dataset 2 - Lugares de emisiones

**Nombre de archivo:** emission_permits_anom_2.json

## 0. Inicialización

Instalar ydata-profiling

In [5]:
!pip install ydata-profiling

Instalar geopy

In [6]:
!pip install geopy

Instalar folium

In [7]:
!pip install folium

Importaciones

In [8]:
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
import seaborn as sns

import folium
import json
from tabulate import tabulate

Visualización de tablas y gráficas

In [9]:
sns.set_style("darkgrid")

def print_table(df):
    print(tabulate(df, headers='keys', tablefmt='simple_outline'))

Lectura y muestra del archivo

In [10]:
# 1. Cargar el JSON
with open('../data/original/emission_permits_anom_2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# 2. Extraer features
features = data['features']

# 3. Construir DataFrame con properties y coordenadas
df = pd.DataFrame([
    {
        **feature['properties'],
        'Latitud': feature['geometry']['coordinates'][1],
        'Longitud': feature['geometry']['coordinates'][0]
    }
    for feature in features
])

df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Cuenca,Latitud,Longitud
0,73640.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,Río Bogotá,4.703418,-74.226561
1,73788.0,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,Resguardo,None,Carbón,Caldera Horno,Río Suárez,5.318407,-73.704281
2,74314.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera Horno,Río Bogotá,4.800462,-74.210355
3,75972.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,Balsillas,None,Fuel Oil No.8,Planta de Asfalto,Río Bogotá,4.678797,-74.284112
4,78824.0,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,El Hato,None,Carbón,Caldera Horno,Río Bogotá,4.699590,-74.193752


**1.** Transformar IDExpediente a integer

In [11]:
df['IDExpediente'] = df['IDExpediente'].astype("Int64")

**2.** Eliminar columnas innecesarias

In [12]:
df = df.drop(columns=['Cuenca'])

**3.** Eliminar duplicados

In [13]:
df = df.drop_duplicates()

**4.** Manejar la capitalización en las variables categóricas

In [14]:
df["Vereda"] = df["Vereda"].str.strip().str.upper()
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].str.strip().str.capitalize()
df.head()

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,None,Otros,Horno,4.703418,-74.226561
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,None,Carbón,Caldera horno,5.318407,-73.704281
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,None,ACPM,Caldera horno,4.800462,-74.210355
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,None,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,None,Carbón,Caldera horno,4.699590,-74.193752


**5.** Mejorar la presentación de los (sin definir)

In [15]:
df["TipoCombustible"] = df["TipoCombustible"].replace("(sin definir)", "Sin definir")
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace("(sin definir)", "Sin definir")
df[(df["TipoCombustible"] == "Sin definir") | (df["TipoFuenteEmision"] == "Sin definir")]

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud
42,138622,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.118908,-73.895435
43,139320,Seguimiento y Control,Ubate,Cundinamarca,TAUSA,RASGATÁ,None,Sin definir,Horno,5.190613,-73.879585
61,150306,Seguimiento y Control,Sabana Centro,Cundinamarca,NEMOCON,PATIO BONITO,None,Sin definir,Sin definir,5.125421,-73.904398
100,171196,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.506914,-74.150135
102,171204,Seguimiento y Control,Bogotá y Municipio de la Calera,Distrito Capital,LOCALIDAD DE CIUDAD BOLIVAR,MOCHUELO BAJO,None,Sin definir,Sin definir,4.516030,-74.149127
...,...,...,...,...,...,...,...,...,...,...,...
539,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,CASCO URBANO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,12.629099,-39.893628
540,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186
541,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,-73.817856
542,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,URBANO,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,11.736503,-41.043145


**6.** Estandarizar el departamento

In [16]:
df["Departamento"] = df["Departamento"].replace({
    "Distrito Capital": "Bogotá",
})

**7.** Estandarizar los municipios

In [17]:
df["Municipio"] = df["Municipio"].replace("LOC.USAQUEN CERROS ORIENTALES", "LOCALIDAD DE USAQUEN")

In [18]:
df["Municipio"] = df["Municipio"].str.replace("LOCALIDAD DE ", "")

In [19]:
print_table(pd.DataFrame(df["Municipio"].value_counts()))

┌─────────────────────┬─────────┐
│ Municipio           │   count │
├─────────────────────┼─────────┤
│ SOACHA              │      56 │
│ CIUDAD BOLIVAR      │      52 │
│ NEMOCON             │      39 │
│ COGUA               │      33 │
│ GIRARDOT            │      25 │
│ CUCUNUBA            │      23 │
│ TAUSA               │      18 │
│ GUACHETA            │      18 │
│ SUTATAUSA           │      15 │
│ RAQUIRA             │      15 │
│ MOSQUERA            │      14 │
│ SIBATE              │      14 │
│ LENGUAZAQUE         │      14 │
│ CAJICA              │      13 │
│ TOCANCIPA           │      12 │
│ FUSAGASUGA          │      12 │
│ VILLAPINZON         │      11 │
│ ZIPAQUIRA           │      10 │
│ COTA                │      10 │
│ FACATATIVA          │       9 │
│ SIMIJACA            │       9 │
│ VILLETA             │       8 │
│ UBATE               │       7 │
│ MADRID              │       7 │
│ RICAURTE            │       6 │
│ CHIQUINQUIRA        │       6 │
│ FUNZA       

**8.** Generar una columna de localidad para las empresas en Bogotá

In [20]:
df["Localidad"] = np.where(df["Departamento"] == "Bogotá", 
                           df["Municipio"], 
                           np.nan)
df["Localidad"]

0                 NaN
1                 NaN
2                 NaN
3                 NaN
4                 NaN
            ...      
539               NaN
540               NaN
541               NaN
542               NaN
543    CIUDAD BOLIVAR
Name: Localidad, Length: 536, dtype: object

**9.** Reemplazar las localidades en la columna municipio, colocando Bogotá

In [21]:
df["Municipio"] = np.where(df["Departamento"] == "Bogotá", 
                           "BOGOTA", 
                           df["Municipio"])
df["Municipio"].value_counts()

Municipio
BOGOTA                 62
SOACHA                 56
NEMOCON                39
COGUA                  33
GIRARDOT               25
CUCUNUBA               23
GUACHETA               18
TAUSA                  18
SUTATAUSA              15
RAQUIRA                15
LENGUAZAQUE            14
SIBATE                 14
MOSQUERA               14
CAJICA                 13
FUSAGASUGA             12
TOCANCIPA              12
VILLAPINZON            11
ZIPAQUIRA              10
COTA                   10
FACATATIVA              9
SIMIJACA                9
VILLETA                 8
UBATE                   7
MADRID                  7
RICAURTE                6
CHIQUINQUIRA            6
FUNZA                   5
SOPO                    4
TENJO                   4
LA CALERA               4
SUESCA                  4
NILO                    3
CARMEN DE CARUPA        3
TOCAIMA                 3
CHOCONTA                3
MANTA                   3
SESQUILE                3
TABIO                   2
AR

**10.** Estandarizar la vereda

In [22]:
df["Vereda"] = df["Vereda"].replace({
    "CASCO URBANO": "AREA URBANA",
    "URBANO": "AREA URBANA",
    "CENTRO URBANO": "AREA URBANA",
})

**11.** Estandarizar los tipos de fuente

In [23]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                            326
Horno                                  131
Caldera                                 24
Caldera horno                           14
Secadores                                7
Planta de asfalto                        6
Molino                                   3
Chimenea 1                               3
Trituradora                              2
Noaplica (área de operación)             2
Reactor                                  1
Planta de asfalto adm                    1
Barrilado de grafito                     1
Filtro molino pendular                   1
Aspiración molino danioni i              1
Triturador de escombros                  1
Campana de extracción  plomo 1           1
Horno de secado                          1
Horno arcillas de soacha tipo túnel      1
Triturador de material                   1
Horno túnel 1 soacha 2                   1
Chimenea triunfo central                 1
700 bhp vr2                         

In [24]:
df["TipoFuenteEmision"] = df["TipoFuenteEmision"].replace({
    "Chimenea 1": "Chimenea",
    "Noaplica (área de operación)": "No aplica (área de operación)",
    "Planta de asfalto adm": "Planta de asfalto",
    "Barrilado de grafito": "Horno",
    "Filtro molino pendular": "Molino",
    "Aspiración molino danioni i": "Molino",
    "Triturador de escombros": "Trituradora",
    "Campana de extracción  plomo 1": "Horno",
    "Horno de secado": "Horno",
    "Horno arcillas de soacha tipo túnel": "Horno",
    "Triturador de material": "Trituradora",
    "Horno túnel 1 soacha 2": "Horno",
    "Chimenea triunfo central": "Chimenea",
    "700 bhp vr2": "Caldera",
    "Planta de mezcla asfáltica": "Planta de asfalto",
    "Batería de coquización a": "Batería de coquización",
    "Molino buhler": "Molino",
    "Planta trituradora": "Trituradora",
    "Batería de producción de coque": "Batería de coquización",
})

In [25]:
df["TipoFuenteEmision"].value_counts()

TipoFuenteEmision
Sin definir                      326
Horno                            136
Caldera                           25
Caldera horno                     14
Planta de asfalto                  8
Secadores                          7
Molino                             6
Trituradora                        5
Chimenea                           4
No aplica (área de operación)      2
Batería de coquización             2
Reactor                            1
Name: count, dtype: int64

**12.** Estandarizar los tipos de combustible

In [27]:
df["TipoCombustible"].value_counts()

TipoCombustible
Sin definir      327
Carbón           128
Otros             26
Gas               20
ACPM              13
NoAplica           7
Fuel Oil No.8      5
Coque              5
Mezcla             2
Madera             1
Leña               1
Hulla              1
Name: count, dtype: int64

In [28]:
df["TipoCombustible"] = df["TipoCombustible"].replace("NoAplica", "No aplica")

In [29]:
df["TipoCombustible"].value_counts()

TipoCombustible
Sin definir      327
Carbón           128
Otros             26
Gas               20
ACPM              13
No aplica          7
Fuel Oil No.8      5
Coque              5
Mezcla             2
Madera             1
Leña               1
Hulla              1
Name: count, dtype: int64

**13.** Llenar nulos en class con 'Sin sanción'

In [30]:
df["Class"] = df["Class"].fillna("Sin sanción")

**14.** Corregir empresas con localización inválida, usando la información geográfica dada

In [31]:
# from geopy.geocoders import Nominatim
# from geopy.extra.rate_limiter import RateLimiter

# # --- 1. Crear geolocalizador ---
# geolocator = Nominatim(user_agent="geo_colombia")
# geocode = RateLimiter(geolocator.geocode, min_delay_seconds=2, max_retries=2, error_wait_seconds=2.0)

# # --- 2. Filtrar las filas mal ubicadas ---
# df_mal_ubicacion = df[(df['Latitud'] >= 12.27) | (df['Longitud'] >= -66.50)].copy()

# print(f"Fuentes con mala ubicación: {len(df_mal_ubicacion)}")

# # --- 3. Función para obtener coordenadas ---
# def obtener_coordenadas_fila(row):
#     # Para Bogotá, usar Localidad si existe; si no, usar Municipio
#     if pd.notna(row['Localidad']):
#         # Es una localidad de Bogotá
#         lugar_vereda = f"{row['Vereda']}, {row['Localidad']}, Bogotá, Colombia"
#         lugar_localidad = f"{row['Localidad']}, Bogotá, Colombia"
#     else:
#         # Es un municipio normal
#         lugar_vereda = f"{row['Vereda']}, {row['Municipio']}, {row['Departamento']}, Colombia"
#         lugar_localidad = f"{row['Municipio']}, {row['Departamento']}, Colombia"
    
#     try:
#         # Intentar primero con vereda
#         location = geocode(lugar_vereda)
#         if location:
#             return pd.Series([location.latitude, location.longitude, "Media"])
#         else:
#             # Si no encuentra la vereda, probar con localidad/municipio
#             location = geocode(lugar_localidad)
#             if location:
#                 if row['Vereda'] == "AREA URBANA":
#                     return pd.Series([location.latitude, location.longitude, "Media"])
#                 else:
#                     return pd.Series([location.latitude, location.longitude, "Baja"])

#     except Exception as e:
#         print(f"Error en {lugar_vereda}: {e}")
    
#     return pd.Series([None, None, None])

# # --- 4. Aplicar función ---
# df_mal_ubicacion[['Latitud', 'Longitud', 'PrecisionUbicacion']] = (
#     df_mal_ubicacion.apply(obtener_coordenadas_fila, axis=1)
# )

# df_mal_ubicacion

Fuentes con mala ubicación: 75


,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud,Localidad,PrecisionUbicacion
19,124002,Seguimiento y Control,Soacha,Cundinamarca,SIBATE,CHACUA,Sin sanción,Otros,Reactor,4.521857,-74.241650,NaN,Media
35,136448,Seguimiento y Control,Soacha,Cundinamarca,SOACHA,AREA URBANA,Sin sanción,Gas,Horno,4.582731,-74.211754,NaN,Media
38,136886,Seguimiento y Control,Soacha,Cundinamarca,SIBATE,CHACUA,Sin sanción,No aplica,Molino,4.521857,-74.241650,NaN,Media
40,137218,Seguimiento y Control,Soacha,Cundinamarca,SIBATE,CHACUA,Sin sanción,Carbón,Caldera,4.521857,-74.241650,NaN,Media
44,139882,Seguimiento y Control,Soacha,Cundinamarca,SOACHA,FUSUNGA,Sin sanción,Carbón,Horno,4.527299,-74.193021,NaN,Media
...,...,...,...,...,...,...,...,...,...,...,...,...,...
536,289234,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,RESGUARDO OCCIDENTE,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.496933,-73.625257,NaN,Baja
537,289754,Sancionatorio,Gualiva,Cundinamarca,VILLETA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.011258,-74.470230,NaN,Media
539,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.496933,-73.625257,NaN,Media
542,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,AREA URBANA,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,4.304638,-74.803074,NaN,Media


In [ ]:
# df_mal_ubicacion.to_csv("archivos_generados/df_mal_ubicacion_corregido.csv")

In [39]:
df_mal_ubicacion = pd.read_csv("archivos_generados/df_mal_ubicacion_corregido.csv", index_col="Unnamed: 0")

In [40]:
import numpy as np
from geopy.distance import distance
from geopy import Point
import pandas as pd

def mover_puntos_geodesico_en_subdf(subdf, desplazamiento_metros=100):
    """
    Para un subdataframe de un Municipio-Vereda, desplaza
    aleatoriamente los puntos que comparten la misma Latitud/Longitud.
    """
    subdf = subdf.copy()

    # Identificar duplicados (solo si Latitud y Longitud no son nulos)
    mask_validos = subdf['Latitud'].notna() & subdf['Longitud'].notna()
    duplicados_mask = subdf[mask_validos].duplicated(subset=["Latitud", "Longitud"], keep=False)

    # si no hay duplicados, devolvemos el subdf sin cambios
    if not duplicados_mask.any():
        return subdf

    # iterar por cada conjunto de coordenadas duplicadas
    indices_duplicados = subdf[mask_validos][duplicados_mask].index
    for (lat0, lon0), grp in subdf.loc[indices_duplicados].groupby(["Latitud", "Longitud"]):
        if grp.shape[0] <= 1:
            continue

        n = grp.shape[0]
        angles = np.random.uniform(0, 360, n)
        radios = np.random.uniform(desplazamiento_metros * 0.3, desplazamiento_metros, n)

        nuevas = []
        for r, a in zip(radios, angles):
            destino = distance(meters=r).destination(Point(lat0, lon0), a)
            nuevas.append((destino.latitude, destino.longitude))

        # asignar nuevas coordenadas al índice correcto
        latitudes = [t[0] for t in nuevas]
        longitudes = [t[1] for t in nuevas]
        subdf.loc[grp.index, "Latitud"] = latitudes
        subdf.loc[grp.index, "Longitud"] = longitudes

    return subdf

# --- Procesar por grupos de Municipio y Vereda sin perder columnas ---
resultado_parts = []

# Cambio clave: usar Localidad cuando existe, si no usar Municipio
df_mal_ubicacion['MunicipioOLocalidad'] = df_mal_ubicacion['Localidad'].fillna(df_mal_ubicacion['Municipio'])

for (mun_loc, ver), sub in df_mal_ubicacion.groupby(["MunicipioOLocalidad", "Vereda"]):
    processed = mover_puntos_geodesico_en_subdf(sub, desplazamiento_metros=100)
    resultado_parts.append(processed)

df_mal_ubicacion = pd.concat(resultado_parts, ignore_index=False)

# Eliminar la columna temporal si quieres
df_mal_ubicacion = df_mal_ubicacion.drop(columns=['MunicipioOLocalidad'])

In [41]:
df_mal_ubicacion

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud,Localidad,PrecisionUbicacion
476,261630,Sancionatorio,Chiquinquira,Boyacá,CHIQUINQUIRA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.618273,-73.816748,NaN,Media
523,282348,Sancionatorio,Bogotá y Municipio de la Calera,Bogotá,BOGOTA,MOCHUELO ALTO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.488357,-74.148341,CIUDAD BOLIVAR,Media
221,281510,En Trámite,Bogotá y Municipio de la Calera,Bogotá,BOGOTA,MOCHUELO BAJO,Sin sanción,Sin definir,Sin definir,4.507812,-74.147559,CIUDAD BOLIVAR,Media
524,282350,Sancionatorio,Bogotá y Municipio de la Calera,Bogotá,BOGOTA,MOCHUELO BAJO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.507802,-74.148226,CIUDAD BOLIVAR,Media
525,282352,Sancionatorio,Bogotá y Municipio de la Calera,Bogotá,BOGOTA,MOCHUELO BAJO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.508354,-74.148910,CIUDAD BOLIVAR,Media
...,...,...,...,...,...,...,...,...,...,...,...,...,...
481,265736,Sancionatorio,Ubate,Cundinamarca,TAUSA,RASGATA ALTO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.194927,-73.956308,NaN,Baja
532,283576,Sancionatorio,Alto Magdalena,Cundinamarca,TOCAIMA,MORRO AZUL,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,4.457927,-74.634223,NaN,Baja
96,170762,Seguimiento y Control,Sabana Centro,Cundinamarca,TOCANCIPA,VERGANZO,Sin sanción,Carbón,Caldera,4.970039,-73.974057,NaN,Media
498,272438,Sancionatorio,Bogotá y Municipio de la Calera,Bogotá,BOGOTA,EL PÁRAMO,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.695219,-74.030932,USAQUEN,Baja


**Nota:** Si presenta el error `Make this Notebook Trusted to load map: File -> Trust Notebook` y usa Anaconda, use la Anaconda Prompt para dirigirse a la carpeta `preparacion` y escriba ```jupyter trust "Dataset 2 - Fuentes de emisiones.ipynb"```

In [42]:
min_lat = df_mal_ubicacion['Latitud'].min()
max_lat = df_mal_ubicacion['Latitud'].max()
min_lon = df_mal_ubicacion['Longitud'].min()
max_lon = df_mal_ubicacion['Longitud'].max()

bounds = [
    [min_lat, min_lon],  # esquina inferior izquierda x
    [min_lat, max_lon],  # inferior derecha
    [max_lat, max_lon],  # superior derecha
    [max_lat, min_lon],  # superior izquierda
]

m = folium.Map(location=[4.71, -74.07], zoom_start=5)

# Dibujar el rectángulo
folium.Polygon(
    locations=bounds,
    color='blue',
    weight=2,
    fill=True,
    fill_opacity=0.1
).add_to(m)

# Agregar marcadores de cada ubicación del dataset
for _, row in df_mal_ubicacion.iterrows():
    color = 'red' if row['PrecisionUbicacion'] == 'Baja' else 'blue'
    folium.Marker(
        location=[row['Latitud'], row['Longitud']],
        tooltip=f"{row['IDExpediente']} - {row['Municipio']}, {row['Vereda']}, Precisión: {row['PrecisionUbicacion']}",
        icon=folium.Icon(color=color)
    ).add_to(m)

# Límites geográficos reales de Colombia
real_max_lat = 12.27
real_max_lon = -66.50

# Marcas adicionales para los límites geográficos de Colombia
folium.PolyLine(
    locations=[[real_max_lat, min_lon], [real_max_lat, real_max_lon]],
    color='green',
    weight=2,
    tooltip=f"Máxima latitud de Colombia terrestre: {real_max_lat}"
).add_to(m)

folium.PolyLine(
    locations=[[min_lat, real_max_lon], [real_max_lat, real_max_lon]],
    color='green',
    weight=2,
    tooltip=f"Mínima longitud de Colombia: {real_max_lon}"
).add_to(m)

m

In [43]:
# --- 5. Crear la columna con valor constante en el df original ---
df['PrecisionUbicacion'] = 'Alta'

# --- 6. Actualizar df original con los valores corregidos ---
cols_to_update = ['Latitud', 'Longitud', 'PrecisionUbicacion']

df.update(df_mal_ubicacion[cols_to_update])
df

,IDExpediente,Estado,Regional,Departamento,Municipio,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud,Localidad,PrecisionUbicacion
0,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,CENTRO,Sin sanción,Otros,Horno,4.703418,-74.226561,NaN,Alta
1,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,RESGUARDO,Sin sanción,Carbón,Caldera horno,5.318407,-73.704281,NaN,Alta
2,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,LA PUNTA,Sin sanción,ACPM,Caldera horno,4.800462,-74.210355,NaN,Alta
3,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,BALSILLAS,Sin sanción,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112,NaN,Alta
4,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,EL HATO,Sin sanción,Carbón,Caldera horno,4.699590,-74.193752,NaN,Alta
...,...,...,...,...,...,...,...,...,...,...,...,...,...
539,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.496933,-73.625257,NaN,Media
540,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186,NaN,Alta
541,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,-73.817856,NaN,Alta
542,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,AREA URBANA,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,4.303999,-74.802852,NaN,Media


**15.** Crear columna identificadora única

In [44]:
df = df.reset_index().rename(columns={'index': 'ID'})
df['ID'] = df['ID'] + 1

In [46]:
df.columns

Index(['ID', 'IDExpediente', 'Estado', 'Regional', 'Departamento', 'Municipio',
       'Vereda', 'Class', 'TipoCombustible', 'TipoFuenteEmision', 'Latitud',
       'Longitud', 'Localidad', 'PrecisionUbicacion'],
      dtype='object')

**16.** Mover columnas

In [48]:
df = df[['ID', 'IDExpediente', 'Estado', 'Regional', 'Departamento', 'Municipio', 'Localidad',
'Vereda', 'Class', 'TipoCombustible', 'TipoFuenteEmision', 'Latitud', 'Longitud', 'PrecisionUbicacion']]

Resultado final

In [49]:
df

,ID,IDExpediente,Estado,Regional,Departamento,Municipio,Localidad,Vereda,Class,TipoCombustible,TipoFuenteEmision,Latitud,Longitud,PrecisionUbicacion
0,1,73640,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,NaN,CENTRO,Sin sanción,Otros,Horno,4.703418,-74.226561,Alta
1,2,73788,Seguimiento y Control,Ubate,Cundinamarca,LENGUAZAQUE,NaN,RESGUARDO,Sin sanción,Carbón,Caldera horno,5.318407,-73.704281,Alta
2,3,74314,Seguimiento y Control,Sabana Occidente,Cundinamarca,MADRID,NaN,LA PUNTA,Sin sanción,ACPM,Caldera horno,4.800462,-74.210355,Alta
3,4,75972,Seguimiento y Control,Sabana Occidente,Cundinamarca,MOSQUERA,NaN,BALSILLAS,Sin sanción,Fuel Oil No.8,Planta de asfalto,4.678797,-74.284112,Alta
4,5,78824,Seguimiento y Control,Sabana Occidente,Cundinamarca,FUNZA,NaN,EL HATO,Sin sanción,Carbón,Caldera horno,4.699590,-74.193752,Alta
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
531,540,291446,Sancionatorio,Chiquinquira,Boyacá,RAQUIRA,NaN,AREA URBANA,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,5.496933,-73.625257,Media
532,541,291712,Sancionatorio,Sumapaz,Cundinamarca,ARBELAEZ,NaN,SAN ROQUE,Por emisiones atmosféricas sin cumplir con los...,Sin definir,Sin definir,4.274013,-74.446186,Alta
533,542,292030,Sancionatorio,Ubate,Cundinamarca,CUCUNUBA,NaN,PUEBLO VIEJO,Por emisiones atmosféricas sin permiso o no cu...,Sin definir,Sin definir,5.228254,-73.817856,Alta
534,543,292322,Sancionatorio,Alto Magdalena,Cundinamarca,GIRARDOT,NaN,AREA URBANA,Emitir por encima de los parámetros establecid...,Sin definir,Sin definir,4.303999,-74.802852,Media


In [50]:
reporte = ProfileReport(df)
reporte.to_file("archivos_generados/Reporte perfilamiento - Dataset 2 Final.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 14/14 [00:00<00:00, 130.05it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Exportar a CSV

In [51]:
df.to_csv('../data/preparada/emission_permits.csv', index=False)